# Producto U1 — Henyelrey Lucio Garcia Chura

**Dimensión:** Tipo de servicio del flujo vs. catálogo IANA de puertos — U1 batch, clasificación

**Rol en el equipo:** Batch / Spark y fuentes externas

**Curso:** Big Data · lambda26 · Proyecto Sello (equipo LLSW3, sección GU)

**Caso:** análisis de tráfico de red del campus universitario (dataset propio, ~400 000 flujos, capturado vía Suricata)

**Metodología:** CRISP-DM — Fases 1 a 5 (hasta modelado/evaluación; el despliegue es Unidad 2)


## Arquitectura Big Data (contexto — no es una fase de CRISP-DM)

Este notebook implementa la **ruta batch** de la arquitectura **Lambda** declarada en el
[Brief técnico-analítico](../../docs/proyecto-sello/brief.md) (Hito S2): capa batch (este
notebook) + capa de velocidad (Kafka, contenido de Unidad 2). El detalle completo de la
decisión Lambda vs. Kappa está en el brief.

## Fase 1 — Comprensión del negocio (CRISP-DM)

**Pregunta de negocio (dimensión propia):** ¿Qué tipo de servicio corresponde a cada flujo
según su comportamiento, comparado con la referencia oficial del catálogo IANA de puertos
conocidos?

**Objetivo de minería de datos:** entrenar un modelo de **clasificación** que prediga la
categoría de servicio a partir de variables de comportamiento del flujo (sin usar el puerto).

**Decisión que habilita:** caracterizar la composición del tráfico del campus por tipo de
servicio, sin depender de que el puerto esté siempre bien declarado.

**Criterio de éxito:** el modelo debe superar con margen claro una línea base ingenua
(predecir siempre la categoría más frecuente) en accuracy y F1 ponderado (referencia
orientativa: al menos 15-20 puntos porcentuales de accuracy por encima de la línea base).


## Fase 2 — Comprensión de los datos (CRISP-DM)

### Extracción con esquema explícito


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType, DoubleType
)
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("u1-producto-henyelrey-servicio")
    .getOrCreate()
)

RUTA_DATOS = "/opt/data/TRCU.csv"

# Hallazgo documentado en S03: con header=True + StructType explícito, Spark asigna
# los campos por POSICION, no por nombre de columna. Si el orden de StructField no
# coincide exactamente con el orden físico del CSV, los valores se corrompen en
# silencio (sin lanzar error). Por eso el orden de abajo respeta el header real
# observado en el dataset, columna por columna.

esquema_flujos = StructType([
    StructField("flow_id", StringType(), True),
    StructField("src_addr", StringType(), True),
    StructField("src_port", IntegerType(), True),
    StructField("dst_addr", StringType(), True),
    StructField("dst_port", IntegerType(), True),
    StructField("ip_prot", IntegerType(), True),
    StructField("timestamp", LongType(), True),
    StructField("flow_duration", DoubleType(), True),
    StructField("down_up_ratio", DoubleType(), True),
    StructField("pkt_len_max", DoubleType(), True),
    StructField("pkt_len_min", DoubleType(), True),
    StructField("pkt_len_mean", DoubleType(), True),
    StructField("pkt_len_var", DoubleType(), True),
    StructField("pkt_len_std", DoubleType(), True),
    StructField("bytes_per_s", DoubleType(), True),
    StructField("pkt_per_s", DoubleType(), True),
    StructField("fwd_pkt_per_s", DoubleType(), True),
    StructField("bwd_pkt_per_s", DoubleType(), True),
    StructField("fwd_pkt_cnt", IntegerType(), True),
    StructField("fwd_pkt_len_tot", DoubleType(), True),
    StructField("fwd_pkt_len_max", DoubleType(), True),
    StructField("fwd_pkt_len_min", DoubleType(), True),
    StructField("fwd_pkt_len_mean", DoubleType(), True),
    StructField("fwd_pkt_len_std", DoubleType(), True),
    StructField("fwd_pkt_hdr_len_tot", IntegerType(), True),
    StructField("fwd_pkt_hdr_len_min", IntegerType(), True),
    StructField("fwd_non_empty_pkt_cnt", IntegerType(), True),
    StructField("bwd_pkt_cnt", IntegerType(), True),
    StructField("bwd_pkt_len_tot", DoubleType(), True),
    StructField("bwd_pkt_len_max", DoubleType(), True),
    StructField("bwd_pkt_len_min", DoubleType(), True),
    StructField("bwd_pkt_len_mean", DoubleType(), True),
    StructField("bwd_pkt_len_std", DoubleType(), True),
    StructField("bwd_pkt_hdr_len_tot", IntegerType(), True),
    StructField("bwd_pkt_hdr_len_min", IntegerType(), True),
    StructField("bwd_non_empty_pkt_cnt", IntegerType(), True),
    StructField("iat_max", DoubleType(), True),
    StructField("iat_min", DoubleType(), True),
    StructField("iat_mean", DoubleType(), True),
    StructField("iat_std", DoubleType(), True),
    StructField("fwd_iat_tot", DoubleType(), True),
    StructField("fwd_iat_max", DoubleType(), True),
    StructField("fwd_iat_min", DoubleType(), True),
    StructField("fwd_iat_mean", DoubleType(), True),
    StructField("fwd_iat_std", DoubleType(), True),
    StructField("bwd_iat_tot", DoubleType(), True),
    StructField("bwd_iat_max", DoubleType(), True),
    StructField("bwd_iat_min", DoubleType(), True),
    StructField("bwd_iat_mean", DoubleType(), True),
    StructField("bwd_iat_std", DoubleType(), True),
    StructField("active_max", DoubleType(), True),
    StructField("active_min", DoubleType(), True),
    StructField("active_mean", DoubleType(), True),
    StructField("active_std", DoubleType(), True),
    StructField("idle_max", DoubleType(), True),
    StructField("idle_min", DoubleType(), True),
    StructField("idle_mean", DoubleType(), True),
    StructField("idle_std", DoubleType(), True),
    StructField("flag_SYN", IntegerType(), True),
    StructField("flag_fin", IntegerType(), True),
    StructField("flag_rst", IntegerType(), True),
    StructField("flag_ack", IntegerType(), True),
    StructField("flag_psh", IntegerType(), True),
    StructField("fwd_flag_psh", IntegerType(), True),
    StructField("bwd_flag_psh", IntegerType(), True),
    StructField("flag_urg", IntegerType(), True),
    StructField("fwd_flag_urg", IntegerType(), True),
    StructField("bwd_flag_urg", IntegerType(), True),
    StructField("flag_cwr", IntegerType(), True),
    StructField("flag_ece", IntegerType(), True),
    StructField("fwd_bulk_bytes_mean", DoubleType(), True),
    StructField("fwd_bulk_pkt_mean", DoubleType(), True),
    StructField("fwd_bulk_rate_mean", DoubleType(), True),
    StructField("bwd_bulk_bytes_mean", DoubleType(), True),
    StructField("bwd_bulk_pkt_mean", DoubleType(), True),
    StructField("bwd_bulk_rate_mean", DoubleType(), True),
    StructField("fwd_subflow_bytes_mean", DoubleType(), True),
    StructField("fwd_subflow_pkt_mean", DoubleType(), True),
    StructField("bwd_subflow_bytes_mean", DoubleType(), True),
    StructField("bwd_subflow_pkt_mean", DoubleType(), True),
    StructField("fwd_tcp_init_win_bytes", IntegerType(), True),
    StructField("bwd_tcp_init_win_bytes", IntegerType(), True),
    StructField("label", StringType(), True),
])

df = spark.read.csv(RUTA_DATOS, header=True, schema=esquema_flujos)

with open(RUTA_DATOS, "r", encoding="utf-8") as f:
    cabecera_real = f.readline().strip().split(",")
assert cabecera_real == [c.name for c in esquema_flujos.fields], (
    "El orden del esquema no coincide con el header real del CSV — revisar antes de continuar."
)

df.printSchema()
df.show(5, truncate=False)
print("Filas totales:", df.count())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/11 02:37:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- flow_id: string (nullable = true)
 |-- src_addr: string (nullable = true)
 |-- src_port: integer (nullable = true)
 |-- dst_addr: string (nullable = true)
 |-- dst_port: integer (nullable = true)
 |-- ip_prot: integer (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- flow_duration: double (nullable = true)
 |-- down_up_ratio: double (nullable = true)
 |-- pkt_len_max: double (nullable = true)
 |-- pkt_len_min: double (nullable = true)
 |-- pkt_len_mean: double (nullable = true)
 |-- pkt_len_var: double (nullable = true)
 |-- pkt_len_std: double (nullable = true)
 |-- bytes_per_s: double (nullable = true)
 |-- pkt_per_s: double (nullable = true)
 |-- fwd_pkt_per_s: double (nullable = true)
 |-- bwd_pkt_per_s: double (nullable = true)
 |-- fwd_pkt_cnt: integer (nullable = true)
 |-- fwd_pkt_len_tot: double (nullable = true)
 |-- fwd_pkt_len_max: double (nullable = true)
 |-- fwd_pkt_len_min: double (nullable = true)
 |-- fwd_pkt_len_mean: double (nullable = true)
 |

26/09/11 02:37:21 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------------------------------------+--------------+--------+--------------+--------+-------+----------------+-------------+-------------+-----------+-----------+------------+-------------+-----------+------------+---------+-------------+-------------+-----------+---------------+---------------+---------------+----------------+---------------+-------------------+-------------------+---------------------+-----------+---------------+---------------+---------------+----------------+---------------+-------------------+-------------------+---------------------+--------+-------+------------+-------------+-----------+-----------+-----------+------------+-------------+-----------+-----------+-----------+------------+-------------+----------+----------+-----------+----------+--------+--------+---------+--------+--------+--------+--------+--------+--------+------------+------------+--------+------------+------------+--------+--------+-------------------+-----------------+------------------

Filas totales: 397354


### Exploración inicial (EDA)


In [2]:
print("Top 15 puertos de destino mas frecuentes:")
df.groupBy("dst_port").count().orderBy(F.desc("count")).show(15)

print("Resumen estadistico de variables de comportamiento:")
df.select("pkt_len_mean", "flow_duration", "iat_mean", "bytes_per_s").describe().show()

print("Nulos por columna clave:")
for columna in ["dst_port", "pkt_len_mean", "flow_duration"]:
    n_nulos = df.filter(F.col(columna).isNull()).count()
    print(f"  {columna}: {n_nulos} nulos")


Top 15 puertos de destino mas frecuentes:


+--------+------+
|dst_port| count|
+--------+------+
|   10001|157470|
|   10002|102610|
|    5355| 32630|
|    5353| 25969|
|    8014|  8621|
|   50160|  4862|
|     137|  4572|
|     138|  3692|
|   56700|  3173|
|     443|  2849|
|   18070|  1886|
|      67|  1875|
|    3289|  1579|
|   57621|  1121|
|   21741|  1088|
+--------+------+
only showing top 15 rows
Resumen estadistico de variables de comportamiento:


+-------+------------------+--------------------+--------------------+-------------------+
|summary|      pkt_len_mean|       flow_duration|            iat_mean|        bytes_per_s|
+-------+------------------+--------------------+--------------------+-------------------+
|  count|            397354|              397354|              397354|             397354|
|   mean|192.97075318445522|1.0620946267620308E7|   3963328.166860908|  550136.9126835192|
| stddev|166.78199958878616| 3.026542849501702E7|1.3829478112778643E7|2.022651824476482E7|
|    min|               0.0|                 0.0|                 0.0|                0.0|
|    max|            1472.0|        1.19999999E8|        1.19999996E8|             2.76E9|
+-------+------------------+--------------------+--------------------+-------------------+

Nulos por columna clave:


  dst_port: 0 nulos


  pkt_len_mean: 0 nulos


  flow_duration: 0 nulos


## Fase 3 — Preparación de los datos (CRISP-DM)

### Transformación y agregación


In [3]:
catalogo_iana = spark.createDataFrame([
    (20, "ftp"), (21, "ftp"), (22, "ssh"), (23, "telnet"), (25, "correo"),
    (53, "dns"), (67, "dhcp"), (68, "dhcp"), (80, "web"), (110, "correo"),
    (123, "ntp"), (143, "correo"), (161, "snmp"), (194, "chat"), (443, "web"),
    (445, "archivos"), (465, "correo"), (587, "correo"), (993, "correo"),
    (995, "correo"), (1433, "bd"), (1521, "bd"), (1723, "vpn"), (3306, "bd"),
    (3389, "escritorio_remoto"), (5432, "bd"), (5900, "escritorio_remoto"),
    (8080, "web"), (8443, "web"),
], ["puerto", "categoria_servicio_iana"])
# Catalogo simplificado de puertos "well-known" (IANA) — servicios mas comunes de campus.
# Registro oficial completo: iana.org/assignments/service-names-port-numbers

df_henyelrey = (
    df.join(catalogo_iana, df.dst_port == catalogo_iana.puerto, "left")
    .withColumn(
        "categoria_servicio_ref",
        F.coalesce(F.col("categoria_servicio_iana"), F.lit("otro_desconocido"))
    )
    .drop("puerto", "categoria_servicio_iana")
)

df_henyelrey.explain(True)

distribucion = df_henyelrey.groupBy("categoria_servicio_ref").agg(F.count("*").alias("n_flujos"))
distribucion.orderBy(F.desc("n_flujos")).show(30, truncate=False)


== Parsed Logical Plan ==
Project [flow_id#0, src_addr#1, src_port#2, dst_addr#3, dst_port#4, ip_prot#5, timestamp#6L, flow_duration#7, down_up_ratio#8, pkt_len_max#9, pkt_len_min#10, pkt_len_mean#11, pkt_len_var#12, pkt_len_std#13, bytes_per_s#14, pkt_per_s#15, fwd_pkt_per_s#16, bwd_pkt_per_s#17, fwd_pkt_cnt#18, fwd_pkt_len_tot#19, fwd_pkt_len_max#20, fwd_pkt_len_min#21, fwd_pkt_len_mean#22, fwd_pkt_len_std#23, fwd_pkt_hdr_len_tot#24, ... 59 more fields]
+- Project [flow_id#0, src_addr#1, src_port#2, dst_addr#3, dst_port#4, ip_prot#5, timestamp#6L, flow_duration#7, down_up_ratio#8, pkt_len_max#9, pkt_len_min#10, pkt_len_mean#11, pkt_len_var#12, pkt_len_std#13, bytes_per_s#14, pkt_per_s#15, fwd_pkt_per_s#16, bwd_pkt_per_s#17, fwd_pkt_cnt#18, fwd_pkt_len_tot#19, fwd_pkt_len_max#20, fwd_pkt_len_min#21, fwd_pkt_len_mean#22, fwd_pkt_len_std#23, fwd_pkt_hdr_len_tot#24, ... 61 more fields]
   +- Join LeftOuter, (cast(dst_port#4 as bigint) = puerto#1185L)
      :- Relation [flow_id#0,src_addr

+----------------------+--------+
|categoria_servicio_ref|n_flujos|
+----------------------+--------+
|otro_desconocido      |390530  |
|web                   |3046    |
|dhcp                  |2745    |
|dns                   |963     |
|snmp                  |43      |
|ntp                   |21      |
|archivos              |6       |
+----------------------+--------+



### Calidad de datos y particionamiento analítico


In [4]:
df_dedup = df_henyelrey.dropDuplicates(["flow_id"])
df_limpio = df_dedup.na.drop(subset=["categoria_servicio_ref"])

RUTA_SALIDA = "/opt/artifacts/henyelrey/flujos_particionado"
(
    df_limpio.write.mode("overwrite")
    .partitionBy("categoria_servicio_ref")
    .parquet(RUTA_SALIDA)
)

df_verificacion = spark.read.parquet(RUTA_SALIDA)
print("Filas tras limpieza:", df_limpio.count())
print("Filas leidas de vuelta desde Parquet:", df_verificacion.count())

df_verificacion.filter(F.col("categoria_servicio_ref") == "web").explain(True)  # PartitionFilters

26/09/11 02:37:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/09/11 02:37:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/11 02:37:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/09/11 02:37:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/09/11 02:37:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/09/11 02:37:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/09/11 02:37:35 WARN MemoryManager: Total allocation exceeds 95.

26/09/11 02:37:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 40.00% for 19 writers
26/09/11 02:37:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 42.22% for 18 writers
26/09/11 02:37:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 44.71% for 17 writers
26/09/11 02:37:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 47.50% for 16 writers
26/09/11 02:37:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 50.67% for 15 writers
26/09/11 02:37:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 54.29% for 14 writers
26/09/11 02:37:36 WARN MemoryManager: Total allocation exceeds 9

26/09/11 02:37:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 40.00% for 19 writers
26/09/11 02:37:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 42.22% for 18 writers
26/09/11 02:37:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 44.71% for 17 writers
26/09/11 02:37:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 47.50% for 16 writers
26/09/11 02:37:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 50.67% for 15 writers
26/09/11 02:37:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 54.29% for 14 writers
26/09/11 02:37:36 WARN MemoryManager: Total allocation exceeds 9

26/09/11 02:37:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/11 02:37:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/09/11 02:37:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/09/11 02:37:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/09/11 02:37:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/09/11 02:37:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 54.29% for 14 writers
26/09/11 02:37:36 WARN MemoryManager: Total allocation exceeds 95

26/09/11 02:37:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/09/11 02:37:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/11 02:37:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/09/11 02:37:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/09/11 02:37:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/09/11 02:37:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/09/11 02:37:37 WARN MemoryManager: Total allocation exceeds 95.

26/09/11 02:37:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/09/11 02:37:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/09/11 02:37:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/09/11 02:37:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/09/11 02:37:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/11 02:37:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Filas tras limpieza: 133311


Filas leidas de vuelta desde Parquet: 133311
== Parsed Logical Plan ==
'Filter '`=`('categoria_servicio_ref, web)
+- Relation [flow_id#2038,src_addr#2039,src_port#2040,dst_addr#2041,dst_port#2042,ip_prot#2043,timestamp#2044L,flow_duration#2045,down_up_ratio#2046,pkt_len_max#2047,pkt_len_min#2048,pkt_len_mean#2049,pkt_len_var#2050,pkt_len_std#2051,bytes_per_s#2052,pkt_per_s#2053,fwd_pkt_per_s#2054,bwd_pkt_per_s#2055,fwd_pkt_cnt#2056,fwd_pkt_len_tot#2057,fwd_pkt_len_max#2058,fwd_pkt_len_min#2059,fwd_pkt_len_mean#2060,fwd_pkt_len_std#2061,fwd_pkt_hdr_len_tot#2062,... 59 more fields] parquet

== Analyzed Logical Plan ==
flow_id: string, src_addr: string, src_port: int, dst_addr: string, dst_port: int, ip_prot: int, timestamp: bigint, flow_duration: double, down_up_ratio: double, pkt_len_max: double, pkt_len_min: double, pkt_len_mean: double, pkt_len_var: double, pkt_len_std: double, bytes_per_s: double, pkt_per_s: double, fwd_pkt_per_s: double, bwd_pkt_per_s: double, fwd_pkt_cnt: int, fwd_

### Selección de predictores y ensamblado del vector de features


In [5]:
from pyspark.ml.feature import VectorAssembler, StringIndexer

# Importante: dst_port/src_port quedan fuera de los predictores a proposito — el objetivo
# es clasificar el servicio a partir del COMPORTAMIENTO del flujo, no leyendo el puerto
# (que es, de hecho, la fuente de la propia etiqueta de referencia).
predictores_henyelrey = [
    "ip_prot", "flow_duration", "pkt_len_mean", "pkt_len_std", "bytes_per_s",
    "fwd_pkt_len_mean", "bwd_pkt_len_mean", "iat_mean", "flag_SYN", "flag_ack",
    "flag_psh", "fwd_tcp_init_win_bytes", "bwd_tcp_init_win_bytes",
]

indexador = StringIndexer(inputCol="categoria_servicio_ref", outputCol="label_idx")
modelo_indexador = indexador.fit(df_limpio)
df_indexado = modelo_indexador.transform(df_limpio)

ensamblador = VectorAssembler(inputCols=predictores_henyelrey, outputCol="features", handleInvalid="skip")
dataset_ml = ensamblador.transform(df_indexado).select("features", "label_idx")

df_train, df_test = dataset_ml.randomSplit([0.8, 0.2], seed=42)
print("Filas de entrenamiento:", df_train.count(), " / Filas de prueba:", df_test.count())


Filas de entrenamiento: 106786  / Filas de prueba: 26525


## Fase 4 — Modelado (CRISP-DM)


In [6]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier

configuraciones_henyelrey = {
    "LogisticRegression base": LogisticRegression(featuresCol="features", labelCol="label_idx"),
    "LogisticRegression + regularizacion": LogisticRegression(featuresCol="features", labelCol="label_idx", regParam=0.1, elasticNetParam=0.5),
    "RandomForestClassifier": RandomForestClassifier(featuresCol="features", labelCol="label_idx", seed=42),
}

modelos_entrenados_henyelrey = {}
predicciones_henyelrey = {}
for nombre, estimador in configuraciones_henyelrey.items():
    modelo = estimador.fit(df_train)
    modelos_entrenados_henyelrey[nombre] = modelo
    predicciones_henyelrey[nombre] = modelo.transform(df_test)
    print("Entrenado:", nombre)


netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory


Entrenado: LogisticRegression base


Entrenado: LogisticRegression + regularizacion


Entrenado: RandomForestClassifier


## Fase 5 — Evaluación (CRISP-DM)


In [7]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

ev_acc = MulticlassClassificationEvaluator(labelCol="label_idx", metricName="accuracy")
ev_f1 = MulticlassClassificationEvaluator(labelCol="label_idx", metricName="f1")
ev_prec = MulticlassClassificationEvaluator(labelCol="label_idx", metricName="weightedPrecision")

clase_mayoritaria = df_test.groupBy("label_idx").count().orderBy(F.desc("count")).first()
frac_mayoritaria = clase_mayoritaria["count"] / df_test.count()
print(f"Linea base ingenua (predecir siempre la clase mayoritaria) -> accuracy={frac_mayoritaria:.4f}\n")

resultados_henyelrey = {}
for nombre, pred in predicciones_henyelrey.items():
    acc = ev_acc.evaluate(pred)
    f1 = ev_f1.evaluate(pred)
    prec = ev_prec.evaluate(pred)
    diferencia_pp = (acc - frac_mayoritaria) * 100
    resultados_henyelrey[nombre] = f1
    print(f"{nombre:38s} Accuracy={acc:.4f}  F1={f1:.4f}  Precision={prec:.4f}  (+{diferencia_pp:.1f} pp vs linea base)")

nombre_ganador = max(resultados_henyelrey, key=resultados_henyelrey.get)
print(f"\nModelo ganador (mayor F1 ponderado): {nombre_ganador}")

# Matriz de confusion del modelo ganador
mejor_pred = predicciones_henyelrey[nombre_ganador]
mejor_pred.groupBy("label_idx", "prediction").count().orderBy("label_idx", "prediction").show(50)

modelo_ganador = modelos_entrenados_henyelrey[nombre_ganador]
modelo_ganador.write().overwrite().save("/opt/artifacts/henyelrey/modelo_servicio")
print("Modelo ganador guardado en /opt/artifacts/henyelrey/modelo_servicio")

Linea base ingenua (predecir siempre la clase mayoritaria) -> accuracy=0.9696



LogisticRegression base                Accuracy=0.9873  F1=0.9865  Precision=0.9860  (+1.8 pp vs linea base)


LogisticRegression + regularizacion    Accuracy=0.9686  F1=0.9551  Precision=0.9457  (+-0.1 pp vs linea base)


RandomForestClassifier                 Accuracy=0.9931  F1=0.9927  Precision=0.9924  (+2.4 pp vs linea base)

Modelo ganador (mayor F1 ponderado): RandomForestClassifier


+---------+----------+-----+
|label_idx|prediction|count|
+---------+----------+-----+
|      0.0|       0.0|25692|
|      0.0|       1.0|   23|
|      0.0|       2.0|    4|
|      1.0|       0.0|   96|
|      1.0|       1.0|  485|
|      1.0|       2.0|    3|
|      2.0|       0.0|    1|
|      2.0|       1.0|   41|
|      2.0|       2.0|  166|
|      3.0|       0.0|    5|
|      4.0|       0.0|    4|
|      5.0|       0.0|    5|
+---------+----------+-----+



Modelo ganador guardado en /opt/artifacts/henyelrey/modelo_servicio


## Cierre de fases y alcance

Este notebook cubre las **5 fases de CRISP-DM hasta el modelado/evaluación**: Comprensión del
negocio → Comprensión de los datos → Preparación de los datos → Modelado → Evaluación.
**No incluye la Fase 6 (Despliegue):** poner el modelo a inferir sobre flujos en vivo es
contenido de Unidad 2 (Spark Structured Streaming + Kafka), declarado como dimensión U2 en
el brief.

## Hallazgo(s) de esta dimensión

Ejecutado de punta a punta contra `TRCU.csv` (397 354 flujos reales capturados por Suricata):

- **Cobertura del catálogo IANA:** el catálogo simplificado (29 puertos *well-known*) solo
  etiqueta al **1.7%** de los flujos en una categoría conocida (6 824 de 397 354: web=3 046,
  dhcp=2 745, dns=963, snmp=43, ntp=21, archivos=6) — el **98.3%** restante cae en
  `otro_desconocido`, porque los puertos reales más frecuentes del tráfico del campus
  (10001, 10002, 5355, 5353, 8014, 50160, 56700, entre otros) no están en el catálogo.
- **Criterio de éxito — NO se cumple:** por esa composición, la línea base ingenua
  (predecir siempre `otro_desconocido`) ya alcanza accuracy=0.9696 en el conjunto de prueba.
  `RandomForestClassifier` gana la comparación (Accuracy=0.9931, F1=0.9927, **+2.4 pp** sobre
  la línea base) frente a `LogisticRegression` (+1.8 pp) y su versión regularizada (-0.1 pp,
  peor que la base), pero **ninguno alcanza el margen de 15-20 pp** definido como criterio de
  éxito en la Fase 1. La matriz de confusión del ganador muestra que sí distingue razonablemente
  bien las dos clases minoritarias con volumen suficiente en el conjunto de prueba — web:
  485/584 correctas (~83%), dhcp: 166/208 correctas (~80%) — pero dns/snmp/ntp tuvieron muy
  pocos casos de prueba (≤5 cada una) para evaluarse con confianza, y el techo de mejora global
  está limitado por cuántos flujos tienen siquiera una categoría de referencia.
- **Recomendación:** ampliar el catálogo IANA (o derivarlo de un registro más completo) antes
  de la Unidad 2, para que la clasificación de tipo de servicio aporte valor real de negocio
  y no quede dominada por la clase `otro_desconocido`.

## Cómo ejecutar este notebook (evidencia de contribución)

1. Levantar el laboratorio (`docker compose up -d` desde `pyspark/`).
2. El dataset real (`TRCU.csv`) ya está en `pyspark/data/`, montado en `/opt/data/` dentro del contenedor — no requiere ajustar `RUTA_DATOS`.
3. Ejecutar de punta a punta (`Run All`), sin intervención manual: el modelo ganador se elige y se guarda automáticamente en la Fase 5.
4. Confirmar la carpeta de salida (`!ls -R` o `os.walk`) sobre `/opt/artifacts/henyelrey/` (Parquet particionado + modelo guardado).
5. Capturar pantalla con reloj del sistema y usuario/perfil visibles.
6. Commit del notebook ejecutado al repositorio del equipo (`pyspark/artifacts/` no se versiona, ver `.gitignore`).